# Instance Matcher Showcase
This is a showcase notebook to highlight the intermediate steps of our instance matcher in-context instance discovery. This work was heavily influenced by "Matcher: Segment Anything with One-Shot Using All-Purpose Feature Matching" by Yang et al.


In [ ]:
# Source - https://stackoverflow.com/a/64166391
# Posted by paxton4416, modified by community. See post 'Timeline' for change history
# Retrieved 2026-03-12, License - CC BY-SA 4.0

%load_ext autoreload
%autoreload 2

# Imports
from models.instance_matcher.instance_matcher import InstanceMatcher
from datasets_fsis import build_fsis_dataset
from dataclasses import dataclass
import plotly.express as px
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as F_vis
import numpy as np
import plotly.io as pio
from PIL import Image

from models.instance_matcher.vision_backbone import DINOv3Backbone, DinoModelType

pio.renderers.default = "notebook_connected"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load the dataset here
@dataclass(frozen=True)
class LvisConfig:
    data_root: str = r"C:\Users\role01-admin\PycharmProjects\SANSA\data"
    fold: int = 0
    shots: int = 1
    multi_train: bool = False,
    ds_weight: list = None

# Dataset
lvis_dataset = build_fsis_dataset("lvis", image_set="val", args=LvisConfig())

In [ ]:
sample = lvis_dataset[np.random.choice(range(len(lvis_dataset)), size=1, replace=False)[0]]
# 1. Pre-process the main image
img_tensor = sample["image"].permute(1, 2, 0).detach().cpu().numpy()
# Robust normalization
img_norm = (img_tensor - img_tensor.min()) / (img_tensor.max() - img_tensor.min())
h, w = img_norm.shape[:2]

# 2. Create a single RGBA canvas for ALL masks
# Initialize as transparent (Alpha = 0)
full_mask_rgba = np.zeros((h, w, 4))

for i, instance in enumerate(sample["instances"]):
    # Get a color from the palette
    color_hex = px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
    # Convert hex to RGB (e.g., "#636EFA" -> [99, 110, 250])
    rgb = [int(color_hex[i:i+2], 16) for i in (1, 3, 5)]

    # Resize mask and convert to numpy
    mask = F.interpolate(instance.unsqueeze(0).unsqueeze(0), size=(h, w)).detach().cpu().numpy()[0, 0]

    # Only update pixels where mask > 0.5 (Binary threshold)
    boolean_mask = mask > 0.5
    full_mask_rgba[boolean_mask, :3] = rgb  # Set RGB
    full_mask_rgba[boolean_mask, 3] = 255    # Set Alpha to opaque for those pixels

# 3. Plot
fig = px.imshow(img_norm) # Base image
fig.add_image(z=full_mask_rgba, opacity=0.5) # One single overlay layer
fig.update_layout(width=800, height=800)
fig.show()

# Instance Matcher
Instance Matcher works in five stages:
1. Forward match patches of the reference masks to other patches in the image to find possible prompts
2. Backword match the prompts and check if they refer back to the reference object. Only keep those that do.
3. Cluster the prompts based on their location with HDBSCAN. The number of clusters that we get, is the number of instances we expect in the image.
4. For each cluster, run SAM on the following prompts:
    1. (Global prompt) The cluster centroid
    2. (Local prompts) All cluster points
5. SAM produces for each prompt type three outputs and we match the best fitting one. The fit is computed again by the similarity of the embeddings.

>[!caution]
> This approach is a single iteration and does not promise to find all instances on the image. It is specificallly bounded by the number of patches in our reference. We can get at most the same amount of prompts back.
## Forward Matching
To find reference points, we embed the image using [DINOv3](https://github.com/facebookresearch/dinov3), then compute the cosine similarity between ref patches and remaining patches.
> [!caution]
> This uses huggingface and gated repository, make sure to be logged in with a valid hf access token via:
> ```
> hf auth login
> ```


In [ ]:
dino_type = DinoModelType.VITS16PLUS  # Select your DINO version
dino = DINOv3Backbone(
    model_type=dino_type,
    image_size=640,
    patch_size=4,
    device=device.type
)

In [ ]:
image = Image.fromarray((img_norm * 255).astype(np.uint8))
embedding = dino.embed_image(
    image,
    keep_dim=False,
)
print(embedding.shape)

In [ ]:
# Initialize our matcher
matcher = InstanceMatcher(
    prompted_backbone=None,
    vision_backbone=None,
    device=device,
)

In [ ]:
# Let's take the first instance as a reference
ref_instance = sample["instances"][0].unsqueeze(0)
ref_instance = F_vis.resize(ref_instance, size=embedding.shape[:-1], interpolation=F_vis.InterpolationMode.NEAREST)
matches = matcher.forward_match(embedding.unsqueeze(0), ref_instance.int())
print(f"Found {len(matches[0])} matches")

In [ ]:
x_ref = matches[0][:, 0].detach().cpu().numpy() * dino.config.patch_size
y_ref = matches[0][:, 1].detach().cpu().numpy() * dino.config.patch_size
x_tgt = matches[0][:, 2].detach().cpu().numpy() * dino.config.patch_size
y_tgt = matches[0][:, 3].detach().cpu().numpy() * dino.config.patch_size

fig = px.imshow(img_norm)
fig.add_scatter(
    x=x_ref,
    y=y_ref,
    marker={"color": "blue", "size": 1},
    mode="markers",
    name="reference",
)
fig.add_scatter(
    x=x_tgt,
    y=y_tgt,
    marker={"color": "red", "size": 5},
    mode="markers",
    name="matches"
)
fig.show()

# Backward Matching
As we can see some matches are random, and have nothing to do with the target category. Hence, we try to match backwards arguing that true matches will point back to the original image.

In [ ]:
filtered_matches = matcher.backward_match(
    embedding.unsqueeze(0),
    ref_instance.to(device),
    matches
)
print(f"Filtered out {len(matches[0]) - len(filtered_matches[0])} matches. {len(matches[0])} -> {len(filtered_matches[0])}")

In [ ]:
x_ref = matches[0][:, 0].detach().cpu().numpy() * dino.config.patch_size
y_ref = matches[0][:, 1].detach().cpu().numpy() * dino.config.patch_size
x_tgt = filtered_matches[0][:, 0].detach().cpu().numpy() * dino.config.patch_size
y_tgt = filtered_matches[0][:, 1].detach().cpu().numpy() * dino.config.patch_size

fig = px.imshow(img_norm)
fig.add_scatter(
    x=x_ref,
    y=y_ref,
    marker={"color": "blue", "size": 1},
    mode="markers",
    name="reference",
)
fig.add_scatter(
    x=x_tgt,
    y=y_tgt,
    marker={"color": "red", "size": 5},
    mode="markers",
    name="filtered matches",
)
fig.show()

## Instance Prompt Sampler
Now we have a set of possible points for prompting. However, we do not know whether these belong to one, two or multiple instances. We need a mapping that clusters them to their objects. Hence we apply a clustering algorithm based on the position in the image (similar to Matcher). Since we do not know of the number of objects, we use a density based clustering technique (DBSCAN).

In [ ]:
batch_instance_prompts = matcher.instance_prompt_samples(filtered_matches)
print(batch_instance_prompts[0])

In [ ]:
fig = px.imshow(img_norm)

for instance_k, instance_points in batch_instance_prompts[0].items():
    fig.add_scatter(
        x=instance_points[:, 0].detach().cpu().numpy() * dino.config.patch_size,
        y=instance_points[:, 1].detach().cpu().numpy() * dino.config.patch_size,
        mode="markers",
        name=f"Cluster {instance_k.item()}",
    )

fig.show()

## Prompting SAM
We now have clusters representing objects (additionally we have a noise cluster that DBSCAN returns, we could treat it as noise or as an instance). Now we need to prompt SAM with the prompts and see, if we can find other instances.